In [6]:
# import torch
# import torch.nn as nn
# from torch.utils.data import DataLoader, TensorDataset
import numpy as np
DATA_PATH = "../data/UCI_dataset/"


SIGNALS = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z'
]

def load_inertial(split):
    X = np.stack([
        np.loadtxt(f"{DATA_PATH}{split}/Inertial Signals/{s}_{split}.txt")
        for s in SIGNALS
    ], axis=2).astype(np.float32)

    y = np.loadtxt(
        f"{DATA_PATH}{split}/y_{split}.txt",
        dtype=int
    ) - 1

    return X, y

X_train_raw, y_train_raw = load_inertial('train')
X_test_raw,  y_test_raw  = load_inertial('test')

mean = X_train_raw.mean(axis=(0,1), keepdims=True)
std  = X_train_raw.std(axis=(0,1),  keepdims=True) + 1e-8
X_train_raw = (X_train_raw - mean) / std
X_test_raw  = (X_test_raw  - mean) / std

print(f"Raw train: {X_train_raw.shape}")  # (7352, 128, 9)
print(f"Raw test:  {X_test_raw.shape}")   # (2947, 128, 9)

Raw train: (7352, 128, 9)
Raw test:  (2947, 128, 9)


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import sys
sys.path.insert(0, '../ISWC22-HAR')

from models.TinyHAR import TinyHAR_Model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model_tinyhar = TinyHAR_Model(
    input_shape=(1, 1, 128, 9),
    number_class=6,
    filter_num=32,
).to(device)

n_params = sum(p.numel() for p in model_tinyhar.parameters())
print(f"Paramètres TinyHAR : {n_params:,}")

Device: cpu
Paramètres TinyHAR : 54,279


In [10]:
X_train = torch.tensor(
    np.expand_dims(X_train_raw, 1),
    dtype=torch.float32
)

X_test = torch.tensor(
    np.expand_dims(X_test_raw, 1),
    dtype=torch.float32
)

y_train = torch.tensor(y_train_raw, dtype=torch.long)
y_test  = torch.tensor(y_test_raw, dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64,
    shuffle=False
)

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_tinyhar.parameters(), lr=1e-3)

for epoch in range(100):

    model_tinyhar.train()

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model_tinyhar(x)

        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()


In [12]:
from sklearn.metrics import classification_report

all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model_tinyhar(x)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

In [13]:
print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.81      0.94      0.87       496
           1       0.92      0.99      0.95       471
           2       0.93      0.96      0.95       420
           3       0.65      0.59      0.62       491
           4       0.73      0.67      0.70       532
           5       0.99      0.91      0.95       537

    accuracy                           0.84      2947
   macro avg       0.84      0.84      0.84      2947
weighted avg       0.84      0.84      0.84      2947

